# **4. XGBoost Modeling**

## **4.1 Import libraries and define paths**

In [1]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [2]:
current_dir = Path.cwd().resolve()
project_dir = current_dir.parent if current_dir.name == "notebooks" else current_dir

# Input + output dir
processed_dir = project_dir / "data" / "processed"
results_dir = project_dir / "results"
models_dir = results_dir / "models"

results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

# from the preprocessing
dataset_path = processed_dir / "era5_supervised_preprocessed.csv"

feature_paths = {
    "baseline": processed_dir / "features_baseline.txt",
    "selected_history": processed_dir / "features_selected_history.txt",
    "full": processed_dir / "features_full.txt",
}

target_column = "extreme_precip_99"

## **4.2 Load processed dataset and feature sets**

In [5]:
# Load the preprocessed dataset and feature set definitions
df = pd.read_csv(dataset_path, parse_dates=["valid_time"])

feature_sets = {
    name: path.read_text(encoding="utf-8").splitlines()
    for name, path in feature_paths.items()
}

# Summary of inputs
print(f"Dataset shape: {df.shape}")
print(f"Time range: {df['valid_time'].min()} to {df['valid_time'].max()}")
print(
    f"Target: {target_column}"
)

pd.DataFrame(
    {
        "feature_set": list(feature_sets.keys()),
        "number_of_features": [len(f) for f in feature_sets.values()],
    }
)

Dataset shape: (131472, 100)
Time range: 2010-01-02 00:00:00 to 2024-12-31 23:00:00
Target: extreme_precip_99


,feature_set,number_of_features
0,baseline,15
1,selected_history,34
2,full,92


## **4.3 Prepare chronological train, validation, and test data**

In [6]:
# reserve 20% for chronological validation for hyperparameter tuning 
train_df = df[df["split"] == "train"].sort_values("valid_time").copy()
test_df = df[df["split"] == "test"].sort_values("valid_time").copy()

split_idx = int(len(train_df) * 0.8)
model_train_df = train_df.iloc[:split_idx].copy()
validation_df = train_df.iloc[split_idx:].copy()

# test for no temporal leakage
assert len(model_train_df) > 0 and len(validation_df) > 0 and len(test_df) > 0, (
    "Empty split"
)
assert model_train_df["valid_time"].max() < validation_df["valid_time"].min(), (
    "Train/validation not chronological"
)
assert validation_df["valid_time"].max() < test_df["valid_time"].min(), (
    "Validation/test not chronological"
)

# summary
splits = {"model_train": model_train_df, "validation": validation_df, "test": test_df}
split_summary = pd.DataFrame(
    {
        name: {
            "n_samples": len(d),
            "n_extreme": int(d[target_column].sum()),
            "extreme_rate_pct": d[target_column].mean() * 100,
            "start": d["valid_time"].min(),
            "end": d["valid_time"].max(),
        }
        for name, d in splits.items()
    }
).T
split_summary

,n_samples,n_extreme,extreme_rate_pct,start,end
model_train,84141,789,0.937712,2010-01-02 00:00:00,2019-08-08 20:00:00
validation,21036,177,0.841415,2019-08-08 21:00:00,2022-01-01 08:00:00
test,26295,349,1.327249,2022-01-01 09:00:00,2024-12-31 23:00:00


## **4.4 Define simple tuning and evaluation functions**

In [ ]:
# Claude by Anthropic (Opus 4.7) was used as a brainstorming assistant and helping to define the structure
# during the development of this code cell. 
# See AI Disclosure Statement submitted with the thesis for details. 

parameter_sets = [
    {"model_variant": "depth_3", "max_depth": 3, "learning_rate": 0.03},
    {"model_variant": "depth_2", "max_depth": 2, "learning_rate": 0.05},
    {"model_variant": "depth_4", "max_depth": 4, "learning_rate": 0.03},
]

BASE_XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 10,
    "n_jobs": -1,
    "tree_method": "hist",
}


def evaluate_binary_classifier(y_true, y_probability, threshold=0.5):
    y_pred = (y_probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_probability),
        "average_precision": average_precision_score(y_true, y_probability),
        "true_negatives": tn,
        "false_positives": fp,
        "false_negatives": fn,
        "true_positives": tp,
    }


def choose_threshold_by_f1(y_true, y_probability):
    thresholds = np.arange(0.05, 0.96, 0.05)
    scores = [
        f1_score(y_true, y_probability >= threshold, zero_division=0)
        for threshold in thresholds
    ]
    return float(thresholds[int(np.argmax(scores))])


def train_xgboost_model(feature_set_name, features):
    X_train, y_train = model_train_df[features], model_train_df[target_column]
    X_valid, y_valid = validation_df[features], validation_df[target_column]
    X_test, y_test = test_df[features], test_df[target_column]

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    tuning_rows, best_model, best_validation_metrics = [], None, None

    for parameters in parameter_sets:
        model_variant = parameters["model_variant"]
        model_parameters = {
            key: value for key, value in parameters.items() if key != "model_variant"
        }

        model = XGBClassifier(
            **BASE_XGB_PARAMS,
            **model_parameters,
            n_estimators=600,
            scale_pos_weight=scale_pos_weight,
            early_stopping_rounds=25,
        )
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

        validation_probability = model.predict_proba(X_valid)[:, 1]
        validation_threshold = choose_threshold_by_f1(y_valid, validation_probability)
        validation_metrics = evaluate_binary_classifier(
            y_valid, validation_probability, validation_threshold
        )
        validation_metrics |= {
            "feature_set": feature_set_name,
            "model_variant": model_variant,
            "number_of_features": len(features),
            "max_depth": parameters["max_depth"],
            "learning_rate": parameters["learning_rate"],
            "best_iteration": model.best_iteration,
            "scale_pos_weight": scale_pos_weight,
        }
        tuning_rows.append(validation_metrics)

        if (
            best_validation_metrics is None
            or validation_metrics["average_precision"]
            > best_validation_metrics["average_precision"]
        ):
            best_model, best_validation_metrics = model, validation_metrics

    final_model_parameters = next(
        {key: value for key, value in parameters.items() if key != "model_variant"}
        for parameters in parameter_sets
        if parameters["model_variant"] == best_validation_metrics["model_variant"]
    )
    final_y_train = train_df[target_column]
    final_scale_pos_weight = (final_y_train == 0).sum() / (final_y_train == 1).sum()

    final_model = XGBClassifier(
        **BASE_XGB_PARAMS,
        **final_model_parameters,
        n_estimators=best_validation_metrics["best_iteration"] + 1,
        scale_pos_weight=final_scale_pos_weight,
    )
    final_model.fit(train_df[features], final_y_train)

    test_probability = final_model.predict_proba(X_test)[:, 1]
    threshold = best_validation_metrics["threshold"]
    test_metrics = evaluate_binary_classifier(y_test, test_probability, threshold)
    test_metrics |= {
        "feature_set": feature_set_name,
        "model_variant": best_validation_metrics["model_variant"],
        "number_of_features": len(features),
        "validation_average_precision": best_validation_metrics["average_precision"],
        "max_depth": best_validation_metrics["max_depth"],
        "learning_rate": best_validation_metrics["learning_rate"],
        "best_iteration": best_validation_metrics["best_iteration"],
        "scale_pos_weight": final_scale_pos_weight,
    }

    predictions = test_df[["valid_time", "tp_mm", target_column]].copy()
    predictions["feature_set"] = feature_set_name
    predictions["predicted_probability"] = test_probability
    predictions["predicted_class"] = (test_probability >= threshold).astype(int)

    return final_model, test_metrics, predictions, tuning_rows

## **4.5 Train all XGBoost models**

In [11]:
models = {}
model_metrics = []
model_predictions = {}
tuning_results = []

for name in feature_sets:
    print(f"Training {name}...")
    model, metrics, predictions, tuning = train_xgboost_model(name, feature_sets[name])
    models[name] = model
    model_metrics.append(metrics)
    model_predictions[name] = predictions
    tuning_results.extend(tuning)
    print(f"  → AP={metrics['average_precision']:.3f}, F1={metrics['f1']:.3f}")

pd.DataFrame(model_metrics)

Training baseline...
  → AP=0.132, F1=0.208
Training selected_history...
  → AP=0.491, F1=0.512
Training full...
  → AP=0.450, F1=0.472


,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,true_negatives,false_positives,false_negatives,true_positives,feature_set,model_variant,number_of_features,validation_average_precision,max_depth,learning_rate,best_iteration,scale_pos_weight
0,0.85,0.973569,0.172348,0.260745,0.207526,0.904852,0.132138,25509,437,258,91,baseline,depth_4,15,0.074665,4,0.03,115,107.878882
1,0.95,0.986461,0.490814,0.535817,0.512329,0.976608,0.491128,25752,194,162,187,selected_history,depth_4,34,0.364006,4,0.03,157,107.878882
2,0.90,0.980909,0.372712,0.641834,0.471579,0.973008,0.450214,25569,377,125,224,full,depth_3,92,0.343069,3,0.03,82,107.878882


## **4.8 Compare model performance**

In [12]:
results_df = pd.DataFrame(model_metrics)
validation_results_df = pd.DataFrame(tuning_results)

column_order = [
    "feature_set",
    "number_of_features",
    "model_variant",
    "validation_average_precision",
    "average_precision",
    "roc_auc",
    "precision",
    "recall",
    "f1",
    "accuracy",
    "true_positives",
    "false_positives",
    "false_negatives",
    "true_negatives",
    "threshold",
    "max_depth",
    "learning_rate",
    "best_iteration",
    "scale_pos_weight",
]

results_df = (
    results_df[column_order]
    .sort_values("average_precision", ascending=False)
    .reset_index(drop=True)
)

validation_results_df = validation_results_df.sort_values(
    ["feature_set", "average_precision"], ascending=[True, False]
).reset_index(drop=True)

results_df

,feature_set,number_of_features,model_variant,validation_average_precision,average_precision,roc_auc,precision,recall,f1,accuracy,true_positives,false_positives,false_negatives,true_negatives,threshold,max_depth,learning_rate,best_iteration,scale_pos_weight
0,selected_history,34,depth_4,0.364006,0.491128,0.976608,0.490814,0.535817,0.512329,0.986461,187,194,162,25752,0.95,4,0.03,157,107.878882
1,full,92,depth_3,0.343069,0.450214,0.973008,0.372712,0.641834,0.471579,0.980909,224,377,125,25569,0.90,3,0.03,82,107.878882
2,baseline,15,depth_4,0.074665,0.132138,0.904852,0.172348,0.260745,0.207526,0.973569,91,437,258,25509,0.85,4,0.03,115,107.878882


## **4.9 Select and save the best model**

In [13]:
best_feature_set = results_df.loc[0, "feature_set"]
best_model = models[best_feature_set]
best_features = feature_sets[best_feature_set]
best_predictions = model_predictions[best_feature_set]

model_path = models_dir / "xgboost_best_model.joblib"
feature_path = results_dir / "xgboost_best_features.csv"
comparison_path = results_dir / "xgboost_model_comparison.csv"
tuning_path = results_dir / "xgboost_validation_tuning_results.csv"
prediction_path = results_dir / "xgboost_best_model_test_predictions.csv"

joblib.dump(best_model, model_path)
pd.Series(best_features, name="feature").to_csv(feature_path, index=False)
results_df.to_csv(comparison_path, index=False)
validation_results_df.to_csv(tuning_path, index=False)
best_predictions.to_csv(prediction_path, index=False)

print(f"Best feature set: {best_feature_set}")
for path in [model_path, feature_path, comparison_path, tuning_path, prediction_path]:
    print(f"  Saved → {path}")

Best feature set: selected_history
  Saved → C:\Users\jensd\Desktop\Data Science\Projects\xai-precipitation-thesis\results\models\xgboost_best_model.joblib
  Saved → C:\Users\jensd\Desktop\Data Science\Projects\xai-precipitation-thesis\results\xgboost_best_features.csv
  Saved → C:\Users\jensd\Desktop\Data Science\Projects\xai-precipitation-thesis\results\xgboost_model_comparison.csv
  Saved → C:\Users\jensd\Desktop\Data Science\Projects\xai-precipitation-thesis\results\xgboost_validation_tuning_results.csv
  Saved → C:\Users\jensd\Desktop\Data Science\Projects\xai-precipitation-thesis\results\xgboost_best_model_test_predictions.csv
